# Lección 10 — Agentes en Producción

La última lección. Acá implementás los 3 pilares que hacen que un agente esté listo para usuarios reales:
1. **Observabilidad** — ver qué hace el agente, cuánto tarda y cuánto cuesta
2. **Evaluación continua** — medir la calidad de respuestas automáticamente
3. **Control de costos** — router de complejidad para usar el modelo justo

In [ ]:
%pip install anthropic python-dotenv -q

In [ ]:
import anthropic
import json
import time
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()
print("Setup listo.")

## Pilar 1 — Observabilidad

Construimos un wrapper que registra automáticamente latencia, tokens usados, herramientas llamadas y errores para cada interacción.

In [ ]:
class ObservabilidadAgente:
    """Registra métricas de todas las interacciones del agente."""
    
    def __init__(self):
        self.registro = []  # historial de todas las interacciones
    
    def ejecutar(self, system: str, mensaje: str, tools: list = None, nombre_sesion: str = "default") -> tuple:
        """Ejecuta una llamada a Claude y registra las métricas."""
        inicio = time.time()
        tools_usadas = []
        error = None
        mensajes = [{"role": "user", "content": mensaje}]
        
        try:
            while True:
                kwargs = {"model": "claude-opus-4-5", "max_tokens": 800, "system": system, "messages": mensajes}
                if tools:
                    kwargs["tools"] = tools
                
                respuesta = client.messages.create(**kwargs)
                
                if tools and respuesta.stop_reason == "tool_use":
                    uso = next(b for b in respuesta.content if b.type == "tool_use")
                    tools_usadas.append(uso.name)
                    mensajes.append({"role": "assistant", "content": respuesta.content})
                    mensajes.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": uso.id, "content": "[resultado simulado]"}]})
                    continue
                
                texto = next(b.text for b in respuesta.content if b.type == "text")
                tokens_entrada = respuesta.usage.input_tokens
                tokens_salida = respuesta.usage.output_tokens
                break
                
        except Exception as e:
            error = str(e)
            texto = f"Error: {error}"
            tokens_entrada = tokens_salida = 0
        
        latencia = round(time.time() - inicio, 2)
        # Precio aproximado claude-opus-4-5: $15/M input, $75/M output
        costo_usd = round((tokens_entrada * 15 + tokens_salida * 75) / 1_000_000, 6)
        
        entrada = {
            "timestamp": datetime.now().strftime("%H:%M:%S"),
            "sesion": nombre_sesion,
            "latencia_s": latencia,
            "tokens_entrada": tokens_entrada,
            "tokens_salida": tokens_salida,
            "costo_usd": costo_usd,
            "tools_usadas": tools_usadas,
            "error": error
        }
        self.registro.append(entrada)
        
        return texto, entrada
    
    def dashboard(self):
        """Muestra un resumen de todas las métricas registradas."""
        if not self.registro:
            print("Sin registros todavía.")
            return
        
        total = len(self.registro)
        errores = sum(1 for r in self.registro if r["error"])
        lat_prom = sum(r["latencia_s"] for r in self.registro) / total
        tokens_total = sum(r["tokens_entrada"] + r["tokens_salida"] for r in self.registro)
        costo_total = sum(r["costo_usd"] for r in self.registro)
        todas_tools = [t for r in self.registro for t in r["tools_usadas"]]
        
        print("\n" + "="*50)
        print("DASHBOARD DE OBSERVABILIDAD")
        print("="*50)
        print(f"Total interacciones:    {total}")
        print(f"Tasa de error:          {errores/total*100:.1f}% ({errores}/{total})")
        print(f"Latencia promedio:      {lat_prom:.2f}s")
        print(f"Latencia máxima:        {max(r['latencia_s'] for r in self.registro):.2f}s")
        print(f"Tokens totales:         {tokens_total:,}")
        print(f"Costo total estimado:   USD {costo_total:.4f}")
        print(f"Costo por interacción:  USD {costo_total/total:.4f}")
        if todas_tools:
            from collections import Counter
            conteo = Counter(todas_tools)
            print(f"Herramientas más usadas: {dict(conteo.most_common(3))}")
        print("="*50)


# Demo
obs = ObservabilidadAgente()

preguntas = [
    "¿Qué es un agente de IA?",
    "¿Cuál es la diferencia entre Claude Opus y Haiku?",
    "¿Cómo instalo la librería de Anthropic en Python?"
]

system = "Sos un asistente de soporte para una comunidad de IA. Respondé en español, de forma concisa."

for i, pregunta in enumerate(preguntas):
    respuesta, metricas = obs.ejecutar(system, pregunta, nombre_sesion=f"demo_{i+1}")
    print(f"[{metricas['timestamp']}] '{pregunta[:40]}...' → {metricas['latencia_s']}s | {metricas['tokens_entrada']+metricas['tokens_salida']} tokens | USD {metricas['costo_usd']}")

obs.dashboard()

## Pilar 2 — Evaluación Continua

Un agente evaluador analiza automáticamente las respuestas del agente principal. Podés correr esto de forma continua en producción para detectar degradaciones de calidad.

In [ ]:
# Dataset de evaluación — casos de prueba conocidos
DATASET_EVALUACION = [
    {
        "pregunta": "¿Qué es Claude Code?",
        "respuesta_esperada_keywords": ["CLI", "terminal", "Anthropic", "programación"],
        "criterio": "Debe mencionar que es una herramienta de línea de comandos de Anthropic para programación"
    },
    {
        "pregunta": "¿Qué modelo de Claude es el más económico?",
        "respuesta_esperada_keywords": ["Haiku", "económico", "rápido", "barato"],
        "criterio": "Debe mencionar Claude Haiku como la opción más económica"
    },
    {
        "pregunta": "¿Cuántos tokens tiene el context window de Claude?",
        "respuesta_esperada_keywords": ["200.000", "200k", "context", "tokens"],
        "criterio": "Debe mencionar aproximadamente 200,000 tokens"
    }
]

def evaluar_con_llm(pregunta: str, respuesta: str, criterio: str) -> dict:
    """Usa Claude para evaluar si una respuesta cumple el criterio."""
    resultado = client.messages.create(
        model="claude-haiku-4-5-20251001",  # modelo barato para evaluación
        max_tokens=200,
        system="Evaluás si respuestas de un chatbot de IA cumplen criterios de calidad. Respondé SOLO JSON: {\"aprobado\": true/false, \"score\": 1-5, \"razon\": \"texto breve\"}",
        messages=[{"role": "user", "content": f"Pregunta: {pregunta}\nCriterio de evaluación: {criterio}\nRespuesta a evaluar: {respuesta}"}]
    )
    texto = resultado.content[0].text.strip()
    if texto.startswith("```"):
        texto = texto.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(texto)


def suite_evaluacion(agente_system: str) -> dict:
    """Corre el dataset de evaluación y devuelve métricas de calidad."""
    
    print("Corriendo suite de evaluación...")
    print("-" * 50)
    
    resultados = []
    
    for caso in DATASET_EVALUACION:
        # Obtener respuesta del agente
        respuesta_agente = client.messages.create(
            model="claude-opus-4-5", max_tokens=300,
            system=agente_system,
            messages=[{"role": "user", "content": caso["pregunta"]}]
        ).content[0].text
        
        # Evaluar con LLM
        evaluacion = evaluar_con_llm(caso["pregunta"], respuesta_agente, caso["criterio"])
        
        resultados.append({
            "pregunta": caso["pregunta"],
            "aprobado": evaluacion["aprobado"],
            "score": evaluacion["score"],
            "razon": evaluacion["razon"]
        })
        
        status = "✓" if evaluacion["aprobado"] else "✗"
        print(f"{status} [{evaluacion['score']}/5] {caso['pregunta'][:45]}")
        if not evaluacion["aprobado"]:
            print(f"   Razón: {evaluacion['razon']}")
    
    aprobados = sum(1 for r in resultados if r["aprobado"])
    score_prom = sum(r["score"] for r in resultados) / len(resultados)
    
    print("-" * 50)
    print(f"Resultado: {aprobados}/{len(resultados)} aprobados | Score promedio: {score_prom:.1f}/5")
    
    return {"aprobados": aprobados, "total": len(resultados), "score_promedio": score_prom, "detalle": resultados}


# Evaluar el agente de soporte
metricas_calidad = suite_evaluacion(
    "Sos un experto en Claude y la API de Anthropic. Respondé preguntas técnicas con precisión en español."
)

## Pilar 3 — Control de Costos con Router de Complejidad

Un modelo pequeño y barato analiza cada consulta y decide si puede responderla o si necesita escalar al modelo más potente. Esto puede reducir costos un 50-70% en consultas simples.

In [ ]:
# Precios aproximados (USD por millón de tokens)
PRECIOS = {
    "claude-haiku-4-5-20251001": {"input": 0.80, "output": 4.00, "nombre": "Haiku (económico)"},
    "claude-opus-4-5": {"input": 15.0, "output": 75.0, "nombre": "Opus (potente)"},
}

def router_complejidad(consulta: str) -> str:
    """Determina qué modelo usar según la complejidad de la consulta."""
    
    clasificacion = client.messages.create(
        model="claude-haiku-4-5-20251001",  # el router siempre usa el modelo barato
        max_tokens=50,
        system="""Clasificás consultas por complejidad. Respondé SOLO con una palabra:
        'simple' — preguntas directas de definición, instalación o uso básico
        'compleja' — análisis, diseño de sistemas, debugging, código avanzado, comparaciones técnicas""",
        messages=[{"role": "user", "content": f"Clasificá: {consulta}"}]
    ).content[0].text.strip().lower()
    
    if "simple" in clasificacion:
        return "claude-haiku-4-5-20251001"
    return "claude-opus-4-5"


def agente_con_router(consulta: str, system: str) -> dict:
    """Agente que elige el modelo según la complejidad de la consulta."""
    
    # El router decide el modelo
    modelo_elegido = router_complejidad(consulta)
    info_modelo = PRECIOS[modelo_elegido]
    
    # Responder con el modelo elegido
    inicio = time.time()
    respuesta_obj = client.messages.create(
        model=modelo_elegido, max_tokens=500,
        system=system,
        messages=[{"role": "user", "content": consulta}]
    )
    latencia = round(time.time() - inicio, 2)
    
    tokens_in = respuesta_obj.usage.input_tokens
    tokens_out = respuesta_obj.usage.output_tokens
    costo = round((tokens_in * info_modelo["input"] + tokens_out * info_modelo["output"]) / 1_000_000, 6)
    
    return {
        "consulta": consulta,
        "modelo": info_modelo["nombre"],
        "respuesta": respuesta_obj.content[0].text,
        "latencia_s": latencia,
        "tokens": tokens_in + tokens_out,
        "costo_usd": costo
    }


# Comparativa: con y sin router
system_soporte = "Sos un experto en Claude Code y agentes de IA. Respondé en español."

consultas_prueba = [
    "¿Qué es la API de Anthropic?",  # simple
    "¿Cuál es el comando para instalar anthropic?",  # simple
    "Diseñá una arquitectura multi-agente para un sistema de soporte al cliente con RAG y metacognición",  # compleja
    "¿Cómo optimizo el uso de tokens en un sistema con 1000 usuarios concurrentes?",  # compleja
]

print("=" * 70)
print("ROUTER DE COMPLEJIDAD — comparativa de costos")
print("=" * 70)

costo_con_router = 0
costo_sin_router = 0
precio_opus = PRECIOS["claude-opus-4-5"]

for consulta in consultas_prueba:
    resultado = agente_con_router(consulta, system_soporte)
    
    # Costo sin router (siempre Opus)
    costo_opus = round(resultado['tokens'] * precio_opus["input"] / 1_000_000, 6)
    
    costo_con_router += resultado['costo_usd']
    costo_sin_router += costo_opus
    
    print(f"\n'{consulta[:50]}'")
    print(f"  → {resultado['modelo']} | {resultado['latencia_s']}s | USD {resultado['costo_usd']} (vs USD {costo_opus} con Opus)")

ahorro = round((1 - costo_con_router/costo_sin_router) * 100, 1)
print(f"\n{'='*70}")
print(f"Costo sin router (todo Opus): USD {costo_sin_router:.6f}")
print(f"Costo con router:             USD {costo_con_router:.6f}")
print(f"Ahorro estimado:              {ahorro}%")
print("="*70)

## Checklist Final — ¿Estás listo para producción?

Revisá cada punto antes de lanzar tu agente a usuarios reales.

In [ ]:
checklist = {
    "Observabilidad": [
        ("Logging de latencia activado", True),
        ("Tracking de tokens y costos", True),
        ("Registro de herramientas usadas", True),
        ("Alertas de errores configuradas", False),  # pendiente
    ],
    "Calidad": [
        ("Dataset de evaluación creado", True),
        ("Suite de tests automatizada", True),
        ("Umbral de calidad definido", True),
        ("Monitoreo en producción", False),  # pendiente
    ],
    "Seguridad": [
        ("Validación de inputs", True),
        ("Límite de herramientas por sesión", True),
        ("Human-in-the-loop para acciones críticas", True),
        ("Rate limiting implementado", False),  # pendiente
    ],
    "Costos": [
        ("Router de complejidad activo", True),
        ("max_tokens limitado", True),
        ("Alertas de gasto configuradas", False),  # pendiente
    ]
}

print("CHECKLIST DE PRODUCCIÓN")
print("=" * 50)
total_items = 0
completados = 0

for categoria, items in checklist.items():
    print(f"\n{categoria}:")
    for descripcion, estado in items:
        total_items += 1
        if estado:
            completados += 1
        icono = "✅" if estado else "⬜"
        print(f"  {icono} {descripcion}")

porcentaje = round(completados/total_items*100)
print(f"\n{'='*50}")
print(f"Progreso: {completados}/{total_items} ({porcentaje}%)")
if porcentaje >= 80:
    print("🟢 Listo para producción")
elif porcentaje >= 60:
    print("🟡 Casi listo — completá los puntos pendientes")
else:
    print("🔴 Necesita más trabajo antes de producción")

## Resumen Final del Curso

| Lección | Concepto | Herramienta principal |
|---|---|---|
| 01 | ¿Qué es un agente? | Anthropic SDK |
| 02 | Frameworks | Clase Agente reutilizable |
| 03 | Patrones de diseño | Instrucciones, JSON, responsabilidad única |
| 04 | Herramientas | Tool schemas, múltiples tools |
| 05 | RAG y memoria | Base de conocimiento, Maker-Checker |
| 06 | Agentes confiables | Validación, Human-in-the-Loop |
| 07 | Planificación | Planificador + Ejecutor |
| 08 | Multi-agente | Secuencial, concurrente, condicional |
| 09 | Metacognición | Fallback, auto-evaluación, feedback |
| 10 | Producción | Observabilidad, evaluación, costos |

---

### ¡Felicitaciones! Completaste el curso.

Pasaste de no saber qué es un agente de IA a poder construir sistemas completos, seguros y listos para producción — todo con Claude.

El siguiente paso es tu propio proyecto. Usá lo que aprendiste, rompé cosas, iterá y compartí tus avances en la comunidad. 🚀